# Detection Pass for Local Review

Runs newspaper-navigator at threshold 0.3 on a stratified sample of pages.
I save the raw detections to JSON, which I download and feed into the local
Tkinter reviewer to mark FPs and FNs.

Setup:
1. colab_upload/ already in Drive from prepare_colab.py
2. Mount Drive, run all cells, download detections.json

### Dependencies

In [ ]:
import subprocess, sys
from pathlib import Path

def sh(cmd): subprocess.run(cmd, shell=True, check=True)

sh("apt-get install -qq -y poppler-utils")
sh(f"{sys.executable} -m pip install -q pdf2image opencv-python-headless pillow scikit-learn tqdm")

try:
    import torch
except ImportError:
    sh(f"{sys.executable} -m pip install -q torch torchvision")

import torch
torch_ver = torch.__version__.split("+")[0]
cuda_tag  = ("cu" + torch.version.cuda.replace(".", "")) if torch.cuda.is_available() else "cpu"
print(f"torch {torch_ver} | {cuda_tag}")

try:
    import detectron2
except ImportError:
    wheel_index = (f"https://dl.fbaipublicfiles.com/detectron2/wheels/"
                   f"{cuda_tag}/torch{torch_ver}/index.html")
    result = subprocess.run(
        f"{sys.executable} -m pip install -q detectron2 -f {wheel_index}", shell=True)
    if result.returncode != 0:
        sh(f"{sys.executable} -m pip install -q --no-build-isolation "
           "'git+https://github.com/facebookresearch/detectron2.git'")

weights_path = Path.home() / "newspaper_navigator_model" / "model_final.pth"
weights_path.parent.mkdir(parents=True, exist_ok=True)
if not weights_path.exists():
    sh(f"wget -q -O {weights_path} "
       "https://github.com/LibraryOfCongress/newspaper-navigator/releases/"
       "download/v1.0.0/model_final.pth")
print("weights ready")

### Mount Drive & Unzip PDFs

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import zipfile
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/ask-maroon/threshold_tune")
UPLOAD_DIR = DRIVE_ROOT / "colab_upload"
PDF_DIR    = DRIVE_ROOT / "pdfs"
PDF_DIR.mkdir(parents=True, exist_ok=True)

zips = sorted(UPLOAD_DIR.glob("pdfs_chunk_*.zip"))
print(f"Found {len(zips)} zip chunk(s)")

for z in zips:
    print(f"  Extracting {z.name} ...")
    with zipfile.ZipFile(z) as zf:
        zf.extractall(PDF_DIR)

pdf_count = len(list(PDF_DIR.rglob("*.pdf")))
print(f"{pdf_count} PDFs available")

### Config

In [ ]:
import torch
from pathlib import Path

DRIVE_ROOT   = Path("/content/drive/MyDrive/ask-maroon/threshold_tune")
PDF_DIR      = DRIVE_ROOT / "pdfs"
MANIFEST_CSV = DRIVE_ROOT / "colab_upload" / "manifest.csv"
OUTPUT_DIR   = DRIVE_ROOT / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DETECTION_THRESH = 0.3   # low pass - review FPs/FNs locally
PAGES_PER_PDF    = 2     # pages sampled per PDF
SEED             = 42

WEIGHTS_PATH = Path.home() / "newspaper_navigator_model" / "model_final.pth"
NUM_CLASSES  = 7
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

IMAGE_CLASS_IDS = {0, 1, 2, 3, 4}
LABEL_NAMES     = ["Photograph", "Illustration", "Map",
                   "Comic", "EditorialCartoon", "Headline", "Advertisement"]

CLASS_COL_MAP = {
    0: "photograph",
    1: "illustration",
    2: "map",
    3: "comic",
    4: "editorial_cartoon",
}

print(f"Device : {DEVICE}")
print(f"Output : {OUTPUT_DIR}")

### Sample Pages from Manifest

In [ ]:
import pandas as pd
import random

manifest = pd.read_csv(MANIFEST_CSV)
rng = random.Random(SEED)

# use all PDFs in manifest
pages = []
for _, row in manifest.iterrows():
    n_pages = 8   # assumed; renderer handles out-of-range gracefully
    for pg in rng.sample(range(n_pages), min(PAGES_PER_PDF, n_pages)):
        pages.append({
            "archive_path": row["archive_path"],
            "year":         str(row["year"]),
            "page_num":     pg,
        })

print(f"Queued {len(pages)} pages from {len(manifest)} PDFs")

### Model Setup

In [ ]:
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2.model_zoo import model_zoo

def build_predictor(score_thresh: float) -> DefaultPredictor:
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(
        "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"))
    cfg.MODEL.WEIGHTS                     = str(WEIGHTS_PATH)
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ROI_HEADS.NUM_CLASSES       = NUM_CLASSES
    cfg.MODEL.DEVICE                      = DEVICE
    cfg.INPUT.MIN_SIZE_TEST               = 800
    cfg.INPUT.MAX_SIZE_TEST               = 1333
    return DefaultPredictor(cfg)

predictor = build_predictor(DETECTION_THRESH)
print(f"Predictor ready (thresh={DETECTION_THRESH}, device={DEVICE})")

### Run Inference & Save Detections

In [ ]:
import json
import numpy as np
from pdf2image import convert_from_path
from tqdm import tqdm

detections = []

for p in tqdm(pages, desc="Inference"):
    pdf_path = PDF_DIR / p["archive_path"]
    pg = p["page_num"]
    try:
        imgs = convert_from_path(str(pdf_path), dpi=150,
                                 first_page=pg + 1, last_page=pg + 1)
        arr = np.array(imgs[0])[:, :, ::-1]   # RGB -> BGR
        out  = predictor(arr)
        inst = out["instances"].to("cpu")
        detections.append({
            "archive_path": p["archive_path"],
            "year":         p["year"],
            "page_num":     pg,
            "img_w":        imgs[0].width,
            "img_h":        imgs[0].height,
            "scores":       inst.scores.numpy().tolist(),
            "classes":      inst.pred_classes.numpy().tolist(),
            "boxes":        inst.pred_boxes.tensor.numpy().tolist(),
        })
    except Exception as e:
        print(f"  [WARN] {p['archive_path']} p{pg}: {e}")
        detections.append({
            "archive_path": p["archive_path"],
            "year":         p["year"],
            "page_num":     pg,
            "img_w":        None,
            "img_h":        None,
            "scores":       [],
            "classes":      [],
            "boxes":        [],
        })

out_path = OUTPUT_DIR / "detections_review.json"
with open(out_path, "w") as f:
    json.dump(detections, f)

print(f"{len(detections)} pages -> detections_review.json")
print("Download this file and place it next to review_labels.py")